<a href="https://colab.research.google.com/github/JCSR2022/challenge3-data-science-Alura/blob/main/data/Data_Preprocessing_Stage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Telecom X – Customer Churn Prediction

## Data Preprocessing Stage

### Project Context

Telecom X faces a critical business challenge: **customer churn**. Customer cancellations directly impact revenue stability, increase acquisition costs, and affect long-term growth.

The objective of this project is to develop **Machine Learning models capable of predicting which customers are most likely to cancel their services** before churn occurs.

This stage represents the transition from **exploratory data analysis to predictive modeling**, where the focus shifts to preparing the dataset to ensure reliable, interpretable, and high-performance models.

Some of this work was made on the Challenge 2 , but in this ocasion the focus will be on billding a pipeline that allow take this process to production.

---

# 🧠 Data Preprocessing Pipeline

The preprocessing stage prepares the raw dataset for Machine Learning modeling by cleaning, transforming, and selecting relevant features.

The following steps will be executed sequentially:


## 1️⃣ Data Loading and check inconsistencies
The first step consists of loading the cleaned dataset generated in the previous exploratory analysis stage.

Key actions include:

- Importing required Python libraries (pandas, numpy, sklearn)

- Loading the dataset into a Pandas DataFrame
Verifying dataset structure

- Check nan, missing, invalid... values

This step ensures the dataset is correctly imported and ready for transformation.

In [1]:
import pandas as pd

In [6]:
url_data = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json"
data_raw = pd.read_json(url_data)
data = pd.json_normalize(data_raw.to_dict(orient="records"))

In [9]:
#display(data.head())
#display(data.info())

In [33]:
#Solve over invalid values
data = data[(data['Churn']=='Yes')|(data['Churn']=='No')]
data['account.Charges.Total'] =  pd.to_numeric(data['account.Charges.Total'], errors='coerce')
data.dropna(subset=['account.Charges.Total'], inplace=True)

In [39]:
#Separate target
X = data.drop('Churn', axis=1)
y = data['Churn']

In [44]:
from sklearn.base import BaseEstimator, TransformerMixin

class Preprocessing(BaseEstimator, TransformerMixin):

    def __init__(self):

      self.categorical_features = ['customer.gender','customer.SeniorCitizen',
      'customer.Partner','customer.Dependents','phone.PhoneService',
      'phone.MultipleLines','internet.OnlineSecurity',
      'internet.OnlineBackup','internet.DeviceProtection','internet.TechSupport',
      'internet.StreamingTV','internet.StreamingMovies',
      'account.PaperlessBilling']

      self.categorical_to_one_hot = ['internet.InternetService','account.Contract','account.PaymentMethod']

      self.numeric_features = ['customer.tenure','account.Charges.Monthly',
                               'account.Charges.Total']


    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        #dont give info for study
        X.drop('customerID', axis=1, inplace=True)

        gender_map = {'Female':0, 'Male':1}
        X['customer.gender'] = X['customer.gender'].map(gender_map)

        Partner_map = {'No':0, 'Yes':1}
        X['customer.Partner'] = X['customer.Partner'].map(Partner_map)

        Dependents_map = {'No':0, 'Yes':1}
        X['customer.Dependents'] = X['customer.Dependents'].map(Dependents_map)

        PhoneService_map = {'No':0, 'Yes':1}
        X['phone.PhoneService'] = X['phone.PhoneService'].map(PhoneService_map)

        MultipleLines_map = {'No':0, 'Yes':1, 'No phone service':0}
        X['phone.MultipleLines'] = X['phone.MultipleLines'].map(MultipleLines_map)

        OnlineSecurity_map = {'No':0, 'Yes':1, 'No internet service':0}
        X['internet.OnlineSecurity'] = X['internet.OnlineSecurity'].map(OnlineSecurity_map)

        OnlineBackup_map = {'No':0, 'Yes':1, 'No internet service':0}
        X['internet.OnlineBackup'] = X['internet.OnlineBackup'].map(OnlineBackup_map)

        DeviceProtection_map = {'No':0, 'Yes':1, 'No internet service':0}
        X['internet.DeviceProtection'] = X['internet.DeviceProtection'].map(DeviceProtection_map)

        TechSupport_map = {'No':0, 'Yes':1, 'No internet service':0}
        X['internet.TechSupport'] = X['internet.TechSupport'].map(TechSupport_map)

        StreamingTV_map = {'No':0, 'Yes':1, 'No internet service':0}
        X['internet.StreamingTV'] = X['internet.StreamingTV'].map(StreamingTV_map)

        StreamingMovies_map = {'No':0, 'Yes':1, 'No internet service':0}
        X['internet.StreamingMovies'] = X['internet.StreamingMovies'].map(StreamingMovies_map)

        PaperlessBilling_map = {'No':0, 'Yes':1}
        X['account.PaperlessBilling'] = X['account.PaperlessBilling'].map(PaperlessBilling_map)


        X.reset_index(drop=True, inplace=True)
        return X

In [47]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


my_preprocessor = Preprocessing()

categorical_features = my_preprocessor.categorical_features
categorical_one_hot = my_preprocessor.categorical_to_one_hot
numeric_features = my_preprocessor.numeric_features

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_one_hot),
        ("num", "passthrough", categorical_features +numeric_features)
    ]
)

In [36]:
for col in categorical_features:
  print(col, data[col].unique())

customer.gender ['Female' 'Male']
customer.SeniorCitizen [0 1]
customer.Partner ['Yes' 'No']
customer.Dependents ['Yes' 'No']
phone.PhoneService ['Yes' 'No']
phone.MultipleLines ['No' 'Yes' 'No phone service']
internet.InternetService ['DSL' 'Fiber optic' 'No']
internet.OnlineSecurity ['No' 'Yes' 'No internet service']
internet.OnlineBackup ['Yes' 'No' 'No internet service']
internet.DeviceProtection ['No' 'Yes' 'No internet service']
internet.TechSupport ['Yes' 'No' 'No internet service']
internet.StreamingTV ['Yes' 'No' 'No internet service']
internet.StreamingMovies ['No' 'Yes' 'No internet service']
account.Contract ['One year' 'Month-to-month' 'Two year']
account.PaperlessBilling ['Yes' 'No']
account.PaymentMethod ['Mailed check' 'Electronic check' 'Credit card (automatic)'
 'Bank transfer (automatic)']


In [40]:
myPreprocessing = Preprocessing()
X_pre = myPreprocessing.fit_transform(X)
X_pre.head()

,customer.gender,customer.SeniorCitizen,customer.Partner,customer.Dependents,customer.tenure,phone.PhoneService,phone.MultipleLines,internet.InternetService,internet.OnlineSecurity,internet.OnlineBackup,internet.DeviceProtection,internet.TechSupport,internet.StreamingTV,internet.StreamingMovies,account.Contract,account.PaperlessBilling,account.PaymentMethod,account.Charges.Monthly,account.Charges.Total
0,0,0,1,1,9,1,0,DSL,0,1,0,1,1,0,One year,1,Mailed check,65.6,593.30
1,1,0,0,0,9,1,1,DSL,0,0,0,0,0,1,Month-to-month,0,Mailed check,59.9,542.40
2,1,0,0,0,4,1,0,Fiber optic,0,0,1,0,0,0,Month-to-month,1,Electronic check,73.9,280.85
3,1,1,1,0,13,1,0,Fiber optic,0,1,1,0,1,1,Month-to-month,1,Electronic check,98.0,1237.85
4,0,1,1,0,3,1,0,Fiber optic,0,0,0,1,1,0,Month-to-month,1,Mailed check,83.9,267.40



## 2️⃣ Data Type and Feature Format Adjustment

Many machine learning algorithms require **numeric input variables** and consistent formatting.

This step includes:

* Converting categorical variables into numerical format
* Ensuring numerical variables have correct data types (`float`, `int`)
* Converting binary categorical variables (Yes/No) into **0/1 encoding**
* Verifying date or tenure-related variables are properly formatted

Proper formatting guarantees compatibility with ML algorithms.

---

In [3]:

data['Churn'].unique()

array(['No', 'Yes', ''], dtype=object)


## 3️⃣ Handling Missing Values

Incomplete data can negatively impact model performance.

Actions performed:

* Detect missing values using `.isnull().sum()`
* Decide an appropriate strategy:

  * Remove rows with excessive missing values
  * Impute values when appropriate (mean, median, or mode)

The goal is to ensure a **complete and reliable dataset**.




---

## 4️⃣ Feature Encoding

Categorical variables must be converted into machine-readable format.

Two main strategies will be applied:

* **Binary Encoding** for Yes/No variables
* **One-Hot Encoding** for multi-category variables such as:

  * Contract type
  * Payment method
  * Internet service type

This prevents algorithms from misinterpreting categorical relationships.

---

## 5️⃣ Feature Scaling (if required)

Certain machine learning models benefit from standardized data.

Scaling may be applied using:

* **Standardization (StandardScaler)**
* **Normalization (MinMaxScaler)**

This step is especially important for algorithms such as:

* Logistic Regression
* Support Vector Machines
* KNN

Tree-based models (Random Forest, Gradient Boosting) are generally insensitive to scaling.

---

## 6️⃣ Feature Selection

To improve model performance and interpretability, irrelevant or weakly informative variables will be removed.

This step includes:

* Correlation analysis with the **target variable (Churn)**
* Removing features with **low predictive value**
* Detecting multicollinearity between predictors
* Retaining variables that contribute meaningful information

Feature selection helps reduce noise and prevent overfitting.

---

## 7️⃣ Train-Test Split

Before model training, the dataset will be separated into:

* **Training Set** (used to train the model)
* **Testing Set** (used to evaluate performance)

Typical split:

```
Train: 80%
Test: 20%
```

Stratified sampling will be applied to preserve the **original churn distribution**.

---

# 🚀 Expected Output of This Stage

At the end of the preprocessing stage, the project will produce:

* A **clean and structured dataset**
* Fully **encoded and formatted features**
* Selected predictive variables
* Training and testing datasets ready for machine learning models

These datasets will be used in the next phase: **Model Development and Evaluation**.

---

# 📈 Next Stage

The next phase will focus on:

* Training predictive models
* Hyperparameter optimization
* Model evaluation
* Feature importance analysis
* Business interpretation of results
